<a href="https://colab.research.google.com/github/claramedina-cmd/AVALIA-O-2/blob/main/C%C3%B3pia_de_avaliacao_1_ml_administracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ALUNAS: Clara Medina e Pietra Santos


# Projeto Prático: Machine Learning & Inteligência de Mercado
## Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)

---

> **Componente Curricular:** Machine Learning aplicado à Administração
> **Instituição:** Curso de Graduação em Administração
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos.

* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** Capítulos 1 a 3 das notas de aula (Ambiente e primeiro contato; Python/pandas/NumPy; Bases de Dados e SQL).

---
# 📝 AVALIAÇÃO 1 (A1) — Machine Learning Aplicado à Administração

| Atributo | Detalhes |
| :--- | :--- |
| **Valor** | 10,0 pontos |
| **Conteúdo cobrado** | Capítulos 1 a 3 (ambiente e primeiro contato; Python, pandas e NumPy; Bases de Dados e SQL) |
| **Formato** | Individual ou em dupla, conforme orientação do professor |
| **Consulta** | Material de aula, internet e uso de IA são permitidos e incentivados — **desde que todo uso de IA seja declarado** (ver regra abaixo) |
| **Entrega** | Repositório público no GitHub, com README, + link enviado no ambiente virtual indicado pelo professor |
| **Prazo** | *(a definir pelo professor — inserir data e horário aqui)* |

## Regras gerais

1. **Não altere a Parte 1** (leitura dos dados). Ela já está pronta e testada — resolva as questões a partir dela.
2. Cada questão tem uma célula de código reservada para a resposta. Adicione quantas células precisar logo abaixo dela, mas **não apague os enunciados**.
3. Onde houver uma pergunta de interpretação (🗣️), responda **em texto, na célula de markdown indicada** — código sozinho não vale a pontuação da interpretação.
4. Se você usar uma IA (Claude, ChatGPT, Gemini etc.) para gerar ou revisar parte do código, **declare isso brevemente** na célula de resposta (ex.: *"Consultei uma IA para revisar a sintaxe do JOIN"*). Não é motivo de desconto — é prática profissional esperada. O que não é aceito é entregar uma resposta que você não entende e não consegue explicar se perguntado.
5. Todas as consultas SQL devem usar `duckdb.sql(...)`, exatamente como praticado em aula.

## Critérios de entrega (obrigatórios — leia antes de começar)

1. Ao terminar, baixe este notebook do Colab: **Arquivo → Fazer download → Download .ipynb**.
2. Crie um repositório público no GitHub (ex.: `ml-administracao-a1-seu-nome`).
3. Suba o notebook (`.ipynb`) para o repositório.
4. Crie um arquivo `README.md` na raiz do repositório (use o modelo fornecido pelo professor).
5. Copie o link do repositório e envie no local indicado pelo professor, dentro do prazo.

> ⚠️ **A entrega via GitHub com README é condição obrigatória para a correção.** Notebooks enviados por e-mail, sem repositório público ou sem README, **não serão corrigidos** até regularização dentro do prazo estipulado pelo professor.

---
## Parte 1 — Leitura dos Dados (fornecida pelo professor)

Esta parte já está pronta. **Apenas execute as células abaixo, na ordem.** Ela baixa o acervo OpenFCS, aplica o filtro de resolução de sete domínios (visto no Capítulo 2) e carrega as três tabelas que vocês vão usar: `edges7` (fato), `entities` e `crosswalk` (dimensões).

In [ ]:
# --- Download e extração do acervo (Capítulo 1) ---
import requests, zipfile, io, os
import pandas as pd

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = "openfcs/openfcs-1.0.0/data/derived/"
print("Download e extracao concluidos.")

Download e extracao concluidos.


In [ ]:
# --- Carregamento das tabelas (Capítulos 1 e 3) ---
edges = pd.read_csv(endereco + "trade_edges.csv")
entities = pd.read_csv(endereco + "entities.csv")
crosswalk = pd.read_csv(endereco + "products_crosswalk.csv")

print(f"trade_edges.csv:        {edges.shape[0]:,} linhas")
print(f"entities.csv:           {entities.shape[0]:,} linhas")
print(f"products_crosswalk.csv: {crosswalk.shape[0]:,} linhas")

trade_edges.csv:        2,197,978 linhas
entities.csv:           309 linhas
products_crosswalk.csv: 14 linhas


In [ ]:
# --- Filtro de resolucao de sete dominios (Capítulo 2) ---
print(edges["resolution"].value_counts())

edges7 = edges[edges["resolution"] == "cer7"].copy()
print(f"\nTabela fato (edges7): {len(edges7):,} linhas")   # deve dar 1.026.400

resolution
craft_sub    1171578
cer7         1026400
Name: count, dtype: int64

Tabela fato (edges7): 1,026,400 linhas


In [ ]:
# --- DuckDB (Capítulo 3) ---
!pip install duckdb --quiet
import duckdb

print("Ambiente pronto. edges7, entities e crosswalk estao carregados.")

Ambiente pronto. edges7, entities e crosswalk estao carregados.


### ✅ Checkpoint
Antes de continuar, confirme:
- `edges7` tem **1.026.400** linhas;
- `edges7.columns` inclui `economy`, `partner`, `year`, `fcs_domain`, `value_usd_millions`, `cer_code`;
- `duckdb.sql("SELECT * FROM edges7 LIMIT 3").df()` roda sem erro.

Se algo não bater, **não prossiga** — revise a Parte 1 antes de ir para as questões.

In [ ]:
# Celula de checkpoint - rode e confira
print(edges7.shape)
print(edges7.columns.tolist())
duckdb.sql("SELECT * FROM edges7 LIMIT 3").df()

(1026400, 11)
['edge_id', 'economy', 'partner', 'product', 'year', 'value_usd_millions', 'flow', 'resolution', 'cer_code', 'fcs_domain', 'mapping_status']


,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
0,1,South America,Other territories,Manufacturing of crafts and design goods,2002,2.026,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
1,8,South America,Other territories,Books and publishing,2002,3.151,Exports,cer7,CER030,D. Books and press,confirmed
2,9,South America,Other territories,"Music, performing and visual arts",2002,0.006,Exports,cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle


---
## Parte 2 — Questões (10,0 pontos)

A partir daqui, o notebook é de vocês. Boa prova!

---
### Questão 1 — Fundamentos e exploração (2,0 pontos)

A partir de `edges7`, faça:

**a) (0,5 pt)** Construa uma **lista** com os nomes únicos dos domínios (`fcs_domain`) presentes na base.

**b) (0,5 pt)** Construa um **dicionário** que associe cada domínio ao número de linhas (fluxos) correspondentes na base.

**c) (1,0 pt) 🗣️** Usando esse dicionário, identifique qual domínio tem mais fluxos registrados. Na célula de markdown abaixo, escreva **uma frase de negócio** explicando o que esse número representa e uma hipótese para o porquê desse domínio ter tantos registros (dica: pense no número de países que participam desse mercado, não apenas no valor exportado).

In [ ]:
# Questao 1a e 1b — ESCREVA SEU CODIGO AQUI



In [ ]:
import pandas as pd

# a) Construir uma lista com os nomes únicos dos domínios (fcs_domain)
unique_fcs_domains = edges7['fcs_domain'].unique().tolist()
print(f"Domínios únicos (lista):\n{unique_fcs_domains}\n")

# b) Construir um dicionário que associe cada domínio ao número de linhas (fluxos)
domain_flow_counts = edges7['fcs_domain'].value_counts().to_dict()
print(f"Contagem de fluxos por domínio (dicionário):\n{domain_flow_counts}\n")

# c) Identificar o domínio com mais fluxos
most_flows_domain = max(domain_flow_counts, key=domain_flow_counts.get)
most_flows_count = domain_flow_counts[most_flows_domain]
print(f"O domínio com mais fluxos registrados é '{most_flows_domain}' com {most_flows_count:,} fluxos.")

Domínios únicos (lista):
['C. Visual arts (crafts) / F. Design', 'D. Books and press', 'B. Performance and celebration / C. Visual arts', 'E. Audiovisual and interactive media', 'F. Design and creative services', 'A. Cultural and natural heritage']

Contagem de fluxos por domínio (dicionário):
{'C. Visual arts (crafts) / F. Design': 316124, 'D. Books and press': 245227, 'E. Audiovisual and interactive media': 198292, 'B. Performance and celebration / C. Visual arts': 175540, 'A. Cultural and natural heritage': 56313, 'F. Design and creative services': 34904}

O domínio com mais fluxos registrados é 'C. Visual arts (crafts) / F. Design' com 316,124 fluxos.


**✍️ Questão 1c — sua resposta em texto aqui:**

O domínio com mais fluxos registrados é 'C. Visual arts (crafts) / F. Design' com 316,124 fluxos. Este número representa a quantidade de registros de transações de exportação/importação para este tipo de bem criativo na base de dados. Uma hipótese para o elevado número de registros neste domínio é a natureza diversificada e descentralizada da produção de bens de artesanato e design, que pode envolver um grande número de pequenos produtores e um vasto leque de países e regiões participando ativamente no comércio global, resultando em muitos fluxos individuais.

---
### Questão 2 — Filtragem, ordenação e agregação com pandas (2,5 pontos)

**a) (1,0 pt)** Filtre `edges7` para obter apenas os fluxos em que o **Brasil** é o país exportador (`economy`), em qualquer domínio ou ano.

**b) (1,0 pt)** A partir desse filtro, use `groupby` para calcular o **valor total exportado pelo Brasil, por domínio** (somando todos os anos).

**c) (0,5 pt) 🗣️** Em qual domínio o Brasil mais exporta, em valor total? Responda em texto, citando o número.

In [ ]:
# Questao 2a e 2b — ESCREVA SEU CODIGO AQUI



In [ ]:
# a) Filtrar para fluxos onde o Brasil é o país exportador
brazil_exports = edges7[edges7['economy'] == 'Brazil']

# b) Calcular o valor total exportado pelo Brasil, por domínio
brazil_exports_by_domain = brazil_exports.groupby('fcs_domain')['value_usd_millions'].sum().sort_values(ascending=False)

print("Valor total exportado pelo Brasil por domínio (em milhões de USD):\n")
print(brazil_exports_by_domain)

Valor total exportado pelo Brasil por domínio (em milhões de USD):

fcs_domain
C. Visual arts (crafts) / F. Design                31724.509
E. Audiovisual and interactive media                3981.583
B. Performance and celebration / C. Visual arts     3108.201
D. Books and press                                  1560.670
A. Cultural and natural heritage                      62.284
F. Design and creative services                        0.796
Name: value_usd_millions, dtype: float64


**✍️ Questão 2c — sua resposta em texto aqui:**

O domínio em que o Brasil mais exporta, em valor total, é 'C. Visual arts (crafts) / F. Design' com 31724.509 milhões de USD. Este valor representa a soma de todas as transações de exportação registradas para este domínio específico, destacando sua relevância na pauta de exportações criativas do Brasil.

---
### Questão 3 — SQL com DuckDB (2,5 pontos)

**a) (1,5 pt)** Escreva uma consulta SQL (via `duckdb.sql(...)`) que retorne, **para o ano de 2023**, o valor total exportado por domínio (`fcs_domain`), ordenado do maior para o menor.

**b) (1,0 pt)** Compare o resultado dessa consulta com o resultado equivalente obtido via `pandas` (`groupby`). Os dois batem? Mostre a comparação no código (não basta afirmar em texto — imprima os dois resultados lado a lado ou calcule a diferença).

In [ ]:
# Questao 3a e 3b — ESCREVA SEU CODIGO AQUI



In [ ]:
# a) Consulta SQL para valor total exportado por domínio em 2023
sql_query = """
SELECT
    fcs_domain,
    SUM(value_usd_millions) AS total_value_usd_millions
FROM
    edges7
WHERE
    year = 2023 AND flow = 'Exports'
GROUP BY
    fcs_domain
ORDER BY
    total_value_usd_millions DESC
"""
sql_result = duckdb.sql(sql_query).df()

print("Resultado da consulta SQL (2023):\n")
display(sql_result)

# b) Resultado equivalente via pandas (groupby) para comparação
pandas_result = edges7[
    (edges7['year'] == 2023) & (edges7['flow'] == 'Exports')
].groupby('fcs_domain')['value_usd_millions'].sum().reset_index()
pandas_result = pandas_result.sort_values(by='value_usd_millions', ascending=False).rename(columns={'value_usd_millions': 'total_value_usd_millions'})

print("\nResultado do pandas (2023):\n")
display(pandas_result)

# Comparação dos resultados
# Arredondar para um número razoável de casas decimais para comparação
sql_result['total_value_usd_millions'] = sql_result['total_value_usd_millions'].round(3)
pandas_result['total_value_usd_millions'] = pandas_result['total_value_usd_millions'].round(3)

# Reordenar o pandas_result para corresponder à ordem do SQL_result para uma comparação justa
pandas_result_sorted = pandas_result.set_index('fcs_domain').loc[sql_result['fcs_domain']].reset_index()

comparison = pd.merge(sql_result, pandas_result_sorted, on='fcs_domain', suffixes=('_sql', '_pandas'))
comparison['difference'] = comparison['total_value_usd_millions_sql'] - comparison['total_value_usd_millions_pandas']

print("\nComparação entre SQL e Pandas:\n")
display(comparison)

if comparison['difference'].abs().sum() < 1e-6:
    print("\nOs resultados da consulta SQL e do pandas batem (diferença insignificante).")
else:
    print("\nHá diferenças entre os resultados da consulta SQL e do pandas.")

Resultado da consulta SQL (2023):



,fcs_domain,total_value_usd_millions
0,C. Visual arts (crafts) / F. Design,993887.877
1,E. Audiovisual and interactive media,154384.619
2,B. Performance and celebration / C. Visual arts,48455.694
3,D. Books and press,34507.589
4,A. Cultural and natural heritage,13734.846
5,F. Design and creative services,124.097



Resultado do pandas (2023):



,fcs_domain,total_value_usd_millions
2,C. Visual arts (crafts) / F. Design,993887.877
4,E. Audiovisual and interactive media,154384.619
1,B. Performance and celebration / C. Visual arts,48455.694
3,D. Books and press,34507.589
0,A. Cultural and natural heritage,13734.846
5,F. Design and creative services,124.097



Comparação entre SQL e Pandas:



,fcs_domain,total_value_usd_millions_sql,total_value_usd_millions_pandas,difference
0,C. Visual arts (crafts) / F. Design,993887.877,993887.877,0.0
1,E. Audiovisual and interactive media,154384.619,154384.619,0.0
2,B. Performance and celebration / C. Visual arts,48455.694,48455.694,0.0
3,D. Books and press,34507.589,34507.589,0.0
4,A. Cultural and natural heritage,13734.846,13734.846,0.0
5,F. Design and creative services,124.097,124.097,0.0



Os resultados da consulta SQL e do pandas batem (diferença insignificante).


---
### Questão 4 — JOIN e modelagem em estrela (3,0 pontos)

**a) (1,0 pt)** Antes de escrever qualquer `JOIN`, inspecione as colunas de `crosswalk` (`.columns.tolist()`) e identifique, em uma célula de markdown, qual coluna representa a chave de produto (equivalente à coluna `cer_code` em `edges7`) e qual coluna representa o status de confirmação do mapeamento.

**b) (1,0 pt)** Escreva uma consulta SQL com `LEFT JOIN` entre `edges7` e `crosswalk` que retorne, para os **10 maiores fluxos de exportação de qualquer domínio em 2024**, também a informação de status de confirmação do mapeamento de produto.

**c) (1,0 pt) 🗣️** Sem escrever código: descreva, em poucas frases, como você desenharia um **esquema em estrela** para responder a esta pergunta de negócio: *"Qual é o parceiro comercial mais importante de cada economia, em cada domínio?"* — Qual seria a tabela fato? Quais dimensões você usaria? Justifique.

**✍️ Questão 4a — colunas identificadas:**

A coluna que representa a chave de produto (equivalente a `cer_code` em `edges7`) é `cer`.
A coluna que representa o status de confirmação do mapeamento é `mapping_status`.

In [ ]:
# Questao 4b — ESCREVA SEU CODIGO AQUI



In [ ]:
# 4b) Consulta SQL com LEFT JOIN entre edges7 e crosswalk
sql_query_4b = """
SELECT
    e.economy,
    e.partner,
    e.fcs_domain,
    e.year,
    e.value_usd_millions,
    c.status
FROM
    edges7 AS e
LEFT JOIN
    crosswalk AS c ON e.cer_code = c.cer
WHERE
    e.year = 2024 AND e.flow = 'Exports'
ORDER BY
    e.value_usd_millions DESC
LIMIT 10
"""

result_4b = duckdb.sql(sql_query_4b).df()
print("Os 10 maiores fluxos de exportação em 2024 com status de mapeamento:\n")
display(result_4b)

Os 10 maiores fluxos de exportação em 2024 com status de mapeamento:



,economy,partner,fcs_domain,year,value_usd_millions,status
0,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2024,71596.783,provisional_straddle
1,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2024,61274.620,provisional_straddle
2,China,United States,C. Visual arts (crafts) / F. Design,2024,45191.825,provisional_straddle
3,G-77 (Group of 77),"China, Hong Kong SAR",C. Visual arts (crafts) / F. Design,2024,18707.251,provisional_straddle
4,China,"China, Hong Kong SAR",C. Visual arts (crafts) / F. Design,2024,12057.264,provisional_straddle
5,G-77 (Group of 77),United Arab Emirates,C. Visual arts (crafts) / F. Design,2024,11877.099,provisional_straddle
6,G-77 (Group of 77),Japan,C. Visual arts (crafts) / F. Design,2024,10183.882,provisional_straddle
7,G-77 (Group of 77),United States,E. Audiovisual and interactive media,2024,9903.889,provisional
8,G-77 (Group of 77),United Kingdom,C. Visual arts (crafts) / F. Design,2024,9835.139,provisional_straddle
9,G-77 (Group of 77),South America,C. Visual arts (crafts) / F. Design,2024,8567.727,provisional_straddle


**✍️ Questão 4c — sua resposta em texto aqui:**

Para a pergunta de negócio "Qual é o parceiro comercial mais importante de cada economia, em cada domínio?" em um esquema em estrela:

*   **Tabela Fato:** Seria a tabela `edges7` (ou uma versão dela). Ela conteria as medidas quantitativas, como `value_usd_millions`, `year`, e as chaves estrangeiras para as dimensões.

*   **Dimensões:**
    *   **Dimensão Tempo:** Com `year` (e, se granularidade maior fosse necessária, mês/dia). Isso permitiria analisar a importância ao longo do tempo.
    *   **Dimensão Economia/País:** Conteria detalhes sobre as economias (`economy` e `partner`), como `name`, `region`, `continent`, etc. Esta dimensão seria usada para filtrar e agrupar pelas economias e parceiros comerciais.
    *   **Dimensão Domínio Criativo:** Conteria informações sobre `fcs_domain`, permitindo segmentar a análise por tipo de bem criativo.

**Justificativa:** Esta estrutura separa os fatos (medidas) das dimensões (contexto), otimizando consultas analíticas. A tabela fato seria pequena e conter medidas, enquanto as dimensões seriam usadas para filtrar e agregar os dados, respondendo eficientemente quem, o quê, onde e quando, no contexto da importância do parceiro comercial (medida pelo valor exportado) para cada economia em cada domínio.

---
## ✅ Checklist final antes de entregar

- [ ] Todas as células rodam em sequência, do início ao fim, sem erro (teste com *Ambiente de execução → Executar tudo*).
- [ ] Todas as perguntas 🗣️ foram respondidas em texto, não só em código.
- [ ] Baixei o notebook (`.ipynb`) e subi para um repositório público no GitHub.
- [ ] Criei um `README.md` no repositório (modelo fornecido pelo professor).
- [ ] Enviei o link do repositório no local indicado, dentro do prazo.

**Boa prova!**